In [1]:
import torch
from torchvision import transforms
from datasets import load_dataset
import cv2
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

dataset = load_dataset("nielsr/CelebA-faces", split="train", cache_dir="./data")

def apply_transform(examples):
    examples["pixel_values"] = [transform(img.convert("RGB")) for img in examples["image"]]
    del examples["image"]
    return examples

dataset.set_transform(apply_transform)

# Linear Noise scheduling 
"""beta_range = torch.linspace(0.0001, 0.02, 1000, device = device)

# parameter calculation
alpha_range = 1 - beta_range
alpha_bar = torch.cumprod(alpha_range, dim = 0)"""

# cosine noise scheduling
T = 1000
s = 0.008

t = torch.linspace(0,T, T+1, device=device)

f_t = torch.cos(((t/T + s)*torch.pi)/((1 + s)*2)) ** 2

alpha_bar = f_t / f_t[0]
beta_range =1 - (alpha_bar[1:] / alpha_bar[:-1])
beta_range = torch.clamp(beta_range, min=0.0001, max=0.9999)

alpha_range = 1.0 - beta_range
alpha_bar = torch.cumprod(alpha_range, dim=0)

# defining the forward process
def forward_process(x_0, t):

    # random gaussian noise to add to the image
    epsilon = torch.randn_like(x_0)

    #reorder alpha_bar to the (B,C,H,W) shape
    a_bar_t = alpha_bar[t].view(-1,1,1,1)
   
    # Simulate a forward pass through a model (for demonstration purposes)
    x_t = (torch.sqrt(a_bar_t)*x_0) + (torch.sqrt(1-a_bar_t)*epsilon)


    return x_t, epsilon

def show_image(tensor):
    # Strict linear mapping from [-1, 1] to [0, 1]
    image = (tensor + 1.0) / 2.0
    image = torch.clamp(image, 0.0, 1.0)
    
    image = image.cpu().permute(1, 2, 0).numpy()
    # Assuming 'img' is your final NumPy image array [0, 255]
    gaussian = cv2.GaussianBlur(image, (0, 0), 2.0)
    sharpened = cv2.addWeighted(image, 1.5, gaussian, -0.5, 0)
    return sharpened

# checking random timestep for random images from the dataset
"""timestamp = torch.tensor([4,50,200,500,850])
indexes = torch.randint(0, 50000 , (5,))
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i,idx in enumerate(indexes):
    x_0 = dataset[idx.item()]["pixel_values"].to(device)
    t = timestamp[i]

    x_t , noise = forward_process(x_0,t)

    axes[0, i].imshow(show_image(x_0))
    axes[0, i].set_title(f"Original")
    axes[0, i].axis("off")

    axes[1, i].imshow(show_image(x_t))
    axes[1, i].set_title(f"t={t.item()}")
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()
print(len(dataset))"""

'timestamp = torch.tensor([4,50,200,500,850])\nindexes = torch.randint(0, 50000 , (5,))\nfig, axes = plt.subplots(2, 5, figsize=(15, 6))\n\nfor i,idx in enumerate(indexes):\n    x_0 = dataset[idx.item()]["pixel_values"].to(device)\n    t = timestamp[i]\n\n    x_t , noise = forward_process(x_0,t)\n\n    axes[0, i].imshow(show_image(x_0))\n    axes[0, i].set_title(f"Original")\n    axes[0, i].axis("off")\n\n    axes[1, i].imshow(show_image(x_t))\n    axes[1, i].set_title(f"t={t.item()}")\n    axes[1, i].axis("off")\n\nplt.tight_layout()\nplt.show()\nprint(len(dataset))'